# Product Bundling Analysis

In [22]:
import pandas as pd
import numpy as np

In [23]:
sales_df = pd.read_csv("pricing_sales_clean.csv")

C:\Users\Dell\AppData\Local\Temp\ipykernel_15920\2627056161.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_df = pd.read_csv("pricing_sales_clean.csv")


In [3]:
# ============================================================
# CREATE PRODUCT BASKETS
# ============================================================

basket_df = (
    sales_df[
        ["Invoice_ID", "Stock_Code"]
    ]
    .drop_duplicates()
    .copy()
)

print(f"Unique invoice-product combinations: {len(basket_df):,}")

display(basket_df.head(10))

Unique invoice-product combinations: 498,373


,Invoice_ID,Stock_Code
0,489434,85048
1,489434,79323P
2,489434,79323W
3,489434,22041
4,489434,21232
5,489434,22064
6,489434,21871
7,489434,21523
8,489435,22350
9,489435,22349


In [4]:
# ============================================================
# GENERATE PRODUCT PAIRS
# ============================================================

pairs = []

for invoice, group in basket_df.groupby("Invoice_ID"):
    products = group["Stock_Code"].unique()

    for i in range(len(products)):
        for j in range(i + 1, len(products)):
            pairs.append([products[i], products[j]])

pairs_df = pd.DataFrame(
    pairs,
    columns=["Product_A", "Product_B"]
)

print("Total pairs:", len(pairs_df))

display(pairs_df.head(10))

Total pairs: 21089322


,Product_A,Product_B
0,85048,79323P
1,85048,79323W
2,85048,22041
3,85048,21232
4,85048,22064
5,85048,21871
6,85048,21523
7,79323P,79323W
8,79323P,22041
9,79323P,21232


In [5]:
pair_counts = (
    pairs_df
    .groupby(["Product_A", "Product_B"])
    .size()
    .reset_index(name="Times_Bought_Together")
)

pair_counts = pair_counts.sort_values(
    "Times_Bought_Together",
    ascending=False
)

display(pair_counts.head(20))

,Product_A,Product_B,Times_Bought_Together
910028,21231,21232,486
1629364,21733,85123A,452
877278,21212,21977,396
879067,21212,84991,388
4871596,84991,21212,383
5038204,85099B,85099F,370
1959570,21931,85099B,359
2822501,22386,85099B,341
4309877,82494L,82482,337
1662073,21754,21755,334


In [6]:
total_invoices = basket_df["Invoice_ID"].nunique()

pair_counts["Support"] = (
    pair_counts["Times_Bought_Together"] / total_invoices
)

display(pair_counts.head(20))

,Product_A,Product_B,Times_Bought_Together,Support
910028,21231,21232,486,0.023195
1629364,21733,85123A,452,0.021572
877278,21212,21977,396,0.018899
879067,21212,84991,388,0.018518
4871596,84991,21212,383,0.018279
5038204,85099B,85099F,370,0.017659
1959570,21931,85099B,359,0.017134
2822501,22386,85099B,341,0.016275
4309877,82494L,82482,337,0.016084
1662073,21754,21755,334,0.015940


In [7]:
product_counts = (
    basket_df
    .groupby("Stock_Code")["Invoice_ID"]
    .nunique()
)

pair_counts["Product_A_Count"] = (
    pair_counts["Product_A"].map(product_counts)
)

pair_counts["Confidence_A_to_B"] = (
    pair_counts["Times_Bought_Together"]
    / pair_counts["Product_A_Count"]
)

display(pair_counts.head(20))

,Product_A,Product_B,Times_Bought_Together,Support,Product_A_Count,Confidence_A_to_B
910028,21231,21232,486,0.023195,1001,0.485514
1629364,21733,85123A,452,0.021572,1066,0.424015
877278,21212,21977,396,0.018899,1853,0.213708
879067,21212,84991,388,0.018518,1853,0.209390
4871596,84991,21212,383,0.018279,1336,0.286677
5038204,85099B,85099F,370,0.017659,1966,0.188199
1959570,21931,85099B,359,0.017134,1179,0.304495
2822501,22386,85099B,341,0.016275,1061,0.321395
4309877,82494L,82482,337,0.016084,1113,0.302785
1662073,21754,21755,334,0.015940,1331,0.250939


In [8]:
product_support = (
    basket_df
    .groupby("Stock_Code")["Invoice_ID"]
    .nunique()
    / total_invoices
)

pair_counts["Support_B"] = (
    pair_counts["Product_B"].map(product_support)
)

pair_counts["Lift"] = (
    pair_counts["Confidence_A_to_B"]
    / pair_counts["Support_B"]
)

display(
    pair_counts
    .sort_values("Lift", ascending=False)
    .head(20)
)

,Product_A,Product_B,Times_Bought_Together,Support,Product_A_Count,Confidence_A_to_B,Support_B,Lift
288617,20721,85171,1,0.000048,1,1.0,0.000048,20953.0
4155534,72751B,72752A,1,0.000048,1,1.0,0.000048,20953.0
3854566,37432,37484,1,0.000048,1,1.0,0.000048,20953.0
1813644,21860,20954,1,0.000048,1,1.0,0.000048,20953.0
1813630,21860,20737,1,0.000048,1,1.0,0.000048,20953.0
4013681,47590a,85132c,1,0.000048,1,1.0,0.000048,20953.0
5370003,90135,90136,1,0.000048,1,1.0,0.000048,20953.0
4772239,84912a,85123b,1,0.000048,1,1.0,0.000048,20953.0
5503406,DCGS0070,DCGS0075,1,0.000048,1,1.0,0.000048,20953.0
4593890,84614C,21772,1,0.000048,1,1.0,0.000048,20953.0


In [9]:
# ============================================================
# FILTER STRONG BUNDLE CANDIDATES
# ============================================================

bundle_candidates = pair_counts[
    (pair_counts["Support"] >= 0.01) &
    (pair_counts["Confidence_A_to_B"] >= 0.20) &
    (pair_counts["Lift"] > 1)
].copy()

bundle_candidates = bundle_candidates.sort_values(
    "Lift",
    ascending=False
)

print(f"Strong bundle candidates: {len(bundle_candidates):,}")

display(bundle_candidates.head(20))

Strong bundle candidates: 74


,Product_A,Product_B,Times_Bought_Together,Support,Product_A_Count,Confidence_A_to_B,Support_B,Lift
3589145,22865,22866,218,0.010404,634,0.343849,0.029256,11.753115
4888155,84997C,84997D,223,0.010643,629,0.354531,0.030258,11.716858
1321636,21531,21535,225,0.010738,539,0.417440,0.036701,11.374011
4885567,84997B,84997C,225,0.010738,669,0.336323,0.030020,11.203455
522568,20971,20972,268,0.012791,656,0.408537,0.038897,10.503150
4837451,84970L,84970S,272,0.012981,643,0.423017,0.043335,9.761539
288778,20723,20724,298,0.014222,678,0.439528,0.049253,8.923867
2752672,22355,22356,316,0.015081,886,0.356659,0.041092,8.679534
2800853,22379,22381,223,0.010643,847,0.263282,0.030592,8.606164
285507,20719,20724,305,0.014556,763,0.399738,0.049253,8.115996


In [10]:
# ============================================================
# ADD REVENUE TO BUNDLE CANDIDATES
# ============================================================

product_revenue = (
    sales_df
    .groupby("Stock_Code")["Revenue"]
    .sum()
)

bundle_candidates["Revenue_A"] = (
    bundle_candidates["Product_A"].map(product_revenue)
)

bundle_candidates["Revenue_B"] = (
    bundle_candidates["Product_B"].map(product_revenue)
)

bundle_candidates["Combined_Revenue"] = (
    bundle_candidates["Revenue_A"]
    + bundle_candidates["Revenue_B"]
)

display(
    bundle_candidates[
        [
            "Product_A",
            "Product_B",
            "Support",
            "Confidence_A_to_B",
            "Lift",
            "Combined_Revenue"
        ]
    ]
    .sort_values("Combined_Revenue", ascending=False)
    .head(20)
)

,Product_A,Product_B,Support,Confidence_A_to_B,Lift,Combined_Revenue
1629364,21733,85123A,0.021572,0.424015,2.707829,186548.40
4840694,84970S,85123A,0.010595,0.244493,1.561375,172712.79
5044278,85099F,85099B,0.013984,0.281731,3.002597,124863.47
2822501,22386,85099B,0.016275,0.321395,3.425324,123624.70
5041252,85099C,85099B,0.010357,0.226042,2.409080,121032.15
1959570,21931,85099B,0.017134,0.304495,3.245214,120470.04
2853108,22411,85099B,0.012552,0.248817,2.651816,118548.58
1953438,21929,85099B,0.010070,0.272610,2.905388,116098.09
2802242,22379,85099B,0.010786,0.266824,2.843726,108731.50
275371,20713,85099B,0.010261,0.314788,3.354907,107308.16


In [11]:
# ============================================================
# FINAL BUNDLE RANKING
# ============================================================

bundle_candidates["Bundle_Score"] = (
    bundle_candidates["Lift"]
    * bundle_candidates["Confidence_A_to_B"]
    * bundle_candidates["Support"]
)

final_bundles = (
    bundle_candidates
    .sort_values("Bundle_Score", ascending=False)
)

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Support",
            "Confidence_A_to_B",
            "Lift",
            "Combined_Revenue",
            "Bundle_Score"
        ]
    ].head(20)
)

,Product_A,Product_B,Support,Confidence_A_to_B,Lift,Combined_Revenue,Bundle_Score
910028,21231,21232,0.023195,0.485514,6.203040,51293.78,0.069855
288778,20723,20724,0.014222,0.439528,8.923867,21876.91,0.055784
522568,20971,20972,0.012791,0.408537,10.503150,31090.64,0.054883
4837451,84970L,84970S,0.012981,0.423017,9.761539,25355.04,0.053604
1321636,21531,21535,0.010738,0.417440,11.374011,17492.79,0.050985
285507,20719,20724,0.014556,0.399738,8.115996,25290.31,0.047225
2752672,22355,22356,0.015081,0.356659,8.679534,27156.19,0.046686
4888155,84997C,84997D,0.010643,0.354531,11.716858,38968.61,0.044210
3589145,22865,22866,0.010404,0.343849,11.753115,31565.90,0.042047
4885567,84997B,84997C,0.010738,0.336323,11.203455,37349.42,0.040462


In [12]:
# ============================================================
# FINAL BUNDLE RECOMMENDATIONS
# ============================================================

final_bundles["Bundle_Recommendation"] = np.where(
    final_bundles["Lift"] >= 2,
    "Strong Bundle Candidate",
    "Consider for Cross-Selling"
)

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Support",
            "Confidence_A_to_B",
            "Lift",
            "Combined_Revenue",
            "Bundle_Score",
            "Bundle_Recommendation"
        ]
    ].head(20)
)

,Product_A,Product_B,Support,Confidence_A_to_B,Lift,Combined_Revenue,Bundle_Score,Bundle_Recommendation
910028,21231,21232,0.023195,0.485514,6.203040,51293.78,0.069855,Strong Bundle Candidate
288778,20723,20724,0.014222,0.439528,8.923867,21876.91,0.055784,Strong Bundle Candidate
522568,20971,20972,0.012791,0.408537,10.503150,31090.64,0.054883,Strong Bundle Candidate
4837451,84970L,84970S,0.012981,0.423017,9.761539,25355.04,0.053604,Strong Bundle Candidate
1321636,21531,21535,0.010738,0.417440,11.374011,17492.79,0.050985,Strong Bundle Candidate
285507,20719,20724,0.014556,0.399738,8.115996,25290.31,0.047225,Strong Bundle Candidate
2752672,22355,22356,0.015081,0.356659,8.679534,27156.19,0.046686,Strong Bundle Candidate
4888155,84997C,84997D,0.010643,0.354531,11.716858,38968.61,0.044210,Strong Bundle Candidate
3589145,22865,22866,0.010404,0.343849,11.753115,31565.90,0.042047,Strong Bundle Candidate
4885567,84997B,84997C,0.010738,0.336323,11.203455,37349.42,0.040462,Strong Bundle Candidate


In [13]:
# ============================================================
# REVIEW TOP BUNDLE CANDIDATES
# ============================================================

top_bundles = final_bundles.head(20).copy()

display(top_bundles)

,Product_A,Product_B,Times_Bought_Together,Support,Product_A_Count,Confidence_A_to_B,Support_B,Lift,Revenue_A,Revenue_B,Combined_Revenue,Bundle_Score,Bundle_Recommendation
910028,21231,21232,486,0.023195,1001,0.485514,0.078270,6.203040,16882.10,34411.68,51293.78,0.069855,Strong Bundle Candidate
288778,20723,20724,298,0.014222,678,0.439528,0.049253,8.923867,8092.07,13784.84,21876.91,0.055784,Strong Bundle Candidate
522568,20971,20972,268,0.012791,656,0.408537,0.038897,10.503150,15138.11,15952.53,31090.64,0.054883,Strong Bundle Candidate
4837451,84970L,84970S,272,0.012981,643,0.423017,0.043335,9.761539,10947.97,14407.07,25355.04,0.053604,Strong Bundle Candidate
1321636,21531,21535,225,0.010738,539,0.417440,0.036701,11.374011,6636.70,10856.09,17492.79,0.050985,Strong Bundle Candidate
285507,20719,20724,305,0.014556,763,0.399738,0.049253,8.115996,11505.47,13784.84,25290.31,0.047225,Strong Bundle Candidate
2752672,22355,22356,316,0.015081,886,0.356659,0.041092,8.679534,17330.84,9825.35,27156.19,0.046686,Strong Bundle Candidate
4888155,84997C,84997D,223,0.010643,629,0.354531,0.030258,11.716858,18445.83,20522.78,38968.61,0.044210,Strong Bundle Candidate
3589145,22865,22866,218,0.010404,634,0.343849,0.029256,11.753115,15608.78,15957.12,31565.90,0.042047,Strong Bundle Candidate
4885567,84997B,84997C,225,0.010738,669,0.336323,0.030020,11.203455,18903.59,18445.83,37349.42,0.040462,Strong Bundle Candidate


In [14]:
# ============================================================
# ASSIGN BUSINESS ACTION TO BUNDLE CANDIDATES
# ============================================================

median_revenue = bundle_candidates["Combined_Revenue"].median()

conditions = [
    (final_bundles["Lift"] >= 2) &
    (final_bundles["Combined_Revenue"] >= median_revenue),

    (final_bundles["Lift"] >= 2)
]

choices = [
    "Test Bundle Offer",
    "Cross-Selling Opportunity"
]

final_bundles["Business_Action"] = np.select(
    conditions,
    choices,
    default="Monitor"
)

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Lift",
            "Combined_Revenue",
            "Bundle_Score",
            "Bundle_Recommendation",
            "Business_Action"
        ]
    ].head(20)
)

,Product_A,Product_B,Lift,Combined_Revenue,Bundle_Score,Bundle_Recommendation,Business_Action
910028,21231,21232,6.203040,51293.78,0.069855,Strong Bundle Candidate,Test Bundle Offer
288778,20723,20724,8.923867,21876.91,0.055784,Strong Bundle Candidate,Cross-Selling Opportunity
522568,20971,20972,10.503150,31090.64,0.054883,Strong Bundle Candidate,Cross-Selling Opportunity
4837451,84970L,84970S,9.761539,25355.04,0.053604,Strong Bundle Candidate,Cross-Selling Opportunity
1321636,21531,21535,11.374011,17492.79,0.050985,Strong Bundle Candidate,Cross-Selling Opportunity
285507,20719,20724,8.115996,25290.31,0.047225,Strong Bundle Candidate,Cross-Selling Opportunity
2752672,22355,22356,8.679534,27156.19,0.046686,Strong Bundle Candidate,Cross-Selling Opportunity
4888155,84997C,84997D,11.716858,38968.61,0.044210,Strong Bundle Candidate,Cross-Selling Opportunity
3589145,22865,22866,11.753115,31565.90,0.042047,Strong Bundle Candidate,Cross-Selling Opportunity
4885567,84997B,84997C,11.203455,37349.42,0.040462,Strong Bundle Candidate,Cross-Selling Opportunity


In [15]:
# ============================================================
# CREATE BUNDLE PRICE & DISCOUNT SCENARIOS
# ============================================================

average_price = (
    sales_df
    .groupby("Stock_Code")["Unit_Price"]
    .mean()
)

final_bundles["Average_Price_A"] = (
    final_bundles["Product_A"].map(average_price)
)

final_bundles["Average_Price_B"] = (
    final_bundles["Product_B"].map(average_price)
)

final_bundles["Bundle_Original_Price"] = (
    final_bundles["Average_Price_A"]
    + final_bundles["Average_Price_B"]
)

final_bundles["Price_5_Discount"] = (
    final_bundles["Bundle_Original_Price"] * 0.95
)

final_bundles["Price_10_Discount"] = (
    final_bundles["Bundle_Original_Price"] * 0.90
)

final_bundles["Price_15_Discount"] = (
    final_bundles["Bundle_Original_Price"] * 0.85
)

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Bundle_Original_Price",
            "Price_5_Discount",
            "Price_10_Discount",
            "Price_15_Discount",
            "Business_Action"
        ]
    ].head(20)
)

,Product_A,Product_B,Bundle_Original_Price,Price_5_Discount,Price_10_Discount,Price_15_Discount,Business_Action
910028,21231,21232,2.935204,2.788444,2.641683,2.494923,Test Bundle Offer
288778,20723,20724,2.137851,2.030958,1.924066,1.817173,Cross-Selling Opportunity
522568,20971,20972,2.800868,2.660825,2.520782,2.380738,Cross-Selling Opportunity
4837451,84970L,84970S,2.272601,2.158971,2.045341,1.931711,Cross-Selling Opportunity
1321636,21531,21535,6.001353,5.701285,5.401218,5.101150,Cross-Selling Opportunity
285507,20719,20724,2.157082,2.049228,1.941373,1.833519,Cross-Selling Opportunity
2752672,22355,22356,2.188863,2.079420,1.969977,1.860534,Cross-Selling Opportunity
4888155,84997C,84997D,7.864764,7.471526,7.078288,6.685050,Cross-Selling Opportunity
3589145,22865,22866,4.654610,4.421880,4.189149,3.956419,Cross-Selling Opportunity
4885567,84997B,84997C,7.821463,7.430390,7.039317,6.648243,Cross-Selling Opportunity


In [16]:
# ============================================================
# CALCULATE DISCOUNT IMPACT
# ============================================================

final_bundles["Discount_Amount_5"] = (
    final_bundles["Bundle_Original_Price"]
    - final_bundles["Price_5_Discount"]
)

final_bundles["Discount_Amount_10"] = (
    final_bundles["Bundle_Original_Price"]
    - final_bundles["Price_10_Discount"]
)

final_bundles["Discount_Amount_15"] = (
    final_bundles["Bundle_Original_Price"]
    - final_bundles["Price_15_Discount"]
)

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Bundle_Original_Price",
            "Discount_Amount_5",
            "Discount_Amount_10",
            "Discount_Amount_15",
            "Business_Action"
        ]
    ].head(20)
)

,Product_A,Product_B,Bundle_Original_Price,Discount_Amount_5,Discount_Amount_10,Discount_Amount_15,Business_Action
910028,21231,21232,2.935204,0.146760,0.293520,0.440281,Test Bundle Offer
288778,20723,20724,2.137851,0.106893,0.213785,0.320678,Cross-Selling Opportunity
522568,20971,20972,2.800868,0.140043,0.280087,0.420130,Cross-Selling Opportunity
4837451,84970L,84970S,2.272601,0.113630,0.227260,0.340890,Cross-Selling Opportunity
1321636,21531,21535,6.001353,0.300068,0.600135,0.900203,Cross-Selling Opportunity
285507,20719,20724,2.157082,0.107854,0.215708,0.323562,Cross-Selling Opportunity
2752672,22355,22356,2.188863,0.109443,0.218886,0.328330,Cross-Selling Opportunity
4888155,84997C,84997D,7.864764,0.393238,0.786476,1.179715,Cross-Selling Opportunity
3589145,22865,22866,4.654610,0.232731,0.465461,0.698192,Cross-Selling Opportunity
4885567,84997B,84997C,7.821463,0.391073,0.782146,1.173219,Cross-Selling Opportunity


In [17]:
# ============================================================
# ESTIMATE BUNDLE REVENUE SCENARIOS
# ============================================================

final_bundles["Revenue_No_Discount"] = (
    final_bundles["Bundle_Original_Price"]
    * final_bundles["Times_Bought_Together"]
)

final_bundles["Revenue_5_Discount"] = (
    final_bundles["Price_5_Discount"]
    * final_bundles["Times_Bought_Together"]
)

final_bundles["Revenue_10_Discount"] = (
    final_bundles["Price_10_Discount"]
    * final_bundles["Times_Bought_Together"]
)

final_bundles["Revenue_15_Discount"] = (
    final_bundles["Price_15_Discount"]
    * final_bundles["Times_Bought_Together"]
)

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Times_Bought_Together",
            "Revenue_No_Discount",
            "Revenue_5_Discount",
            "Revenue_10_Discount",
            "Revenue_15_Discount"
        ]
    ].head(20)
)

,Product_A,Product_B,Times_Bought_Together,Revenue_No_Discount,Revenue_5_Discount,Revenue_10_Discount,Revenue_15_Discount
910028,21231,21232,486,1426.509027,1355.183576,1283.858124,1212.532673
288778,20723,20724,298,637.079611,605.225631,573.371650,541.517670
522568,20971,20972,268,750.632727,713.101090,675.569454,638.037818
4837451,84970L,84970S,272,618.147572,587.240194,556.332815,525.425436
1321636,21531,21535,225,1350.304394,1282.789174,1215.273955,1147.758735
285507,20719,20724,305,657.909901,625.014406,592.118911,559.223416
2752672,22355,22356,316,691.680849,657.096807,622.512765,587.928722
4888155,84997C,84997D,223,1753.842476,1666.150352,1578.458229,1490.766105
3589145,22865,22866,218,1014.705053,963.969800,913.234547,862.499295
4885567,84997B,84997C,225,1759.829145,1671.837687,1583.846230,1495.854773


In [18]:
# ============================================================
# REQUIRED EXTRA SALES TO MAINTAIN REVENUE
# ============================================================

final_bundles["Required_Sales_5"] = (
    final_bundles["Revenue_No_Discount"]
    / final_bundles["Price_5_Discount"]
)

final_bundles["Required_Sales_10"] = (
    final_bundles["Revenue_No_Discount"]
    / final_bundles["Price_10_Discount"]
)

final_bundles["Required_Sales_15"] = (
    final_bundles["Revenue_No_Discount"]
    / final_bundles["Price_15_Discount"]
)

# Extra sales needed compared to current bundle purchases

final_bundles["Extra_Sales_5"] = (
    final_bundles["Required_Sales_5"]
    - final_bundles["Times_Bought_Together"]
)

final_bundles["Extra_Sales_10"] = (
    final_bundles["Required_Sales_10"]
    - final_bundles["Times_Bought_Together"]
)

final_bundles["Extra_Sales_15"] = (
    final_bundles["Required_Sales_15"]
    - final_bundles["Times_Bought_Together"]
)

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Times_Bought_Together",
            "Extra_Sales_5",
            "Extra_Sales_10",
            "Extra_Sales_15"
        ]
    ].head(20)
)

,Product_A,Product_B,Times_Bought_Together,Extra_Sales_5,Extra_Sales_10,Extra_Sales_15
910028,21231,21232,486,25.578947,54.000000,85.764706
288778,20723,20724,298,15.684211,33.111111,52.588235
522568,20971,20972,268,14.105263,29.777778,47.294118
4837451,84970L,84970S,272,14.315789,30.222222,48.000000
1321636,21531,21535,225,11.842105,25.000000,39.705882
285507,20719,20724,305,16.052632,33.888889,53.823529
2752672,22355,22356,316,16.631579,35.111111,55.764706
4888155,84997C,84997D,223,11.736842,24.777778,39.352941
3589145,22865,22866,218,11.473684,24.222222,38.470588
4885567,84997B,84997C,225,11.842105,25.000000,39.705882


In [19]:
# ============================================================
# REQUIRED SALES GROWTH PERCENTAGE
# ============================================================

final_bundles["Growth_Needed_5_Percent"] = (
    final_bundles["Extra_Sales_5"]
    / final_bundles["Times_Bought_Together"]
) * 100

final_bundles["Growth_Needed_10_Percent"] = (
    final_bundles["Extra_Sales_10"]
    / final_bundles["Times_Bought_Together"]
) * 100

final_bundles["Growth_Needed_15_Percent"] = (
    final_bundles["Extra_Sales_15"]
    / final_bundles["Times_Bought_Together"]
) * 100

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Growth_Needed_5_Percent",
            "Growth_Needed_10_Percent",
            "Growth_Needed_15_Percent"
        ]
    ].head(20)
)

,Product_A,Product_B,Growth_Needed_5_Percent,Growth_Needed_10_Percent,Growth_Needed_15_Percent
910028,21231,21232,5.263158,11.111111,17.647059
288778,20723,20724,5.263158,11.111111,17.647059
522568,20971,20972,5.263158,11.111111,17.647059
4837451,84970L,84970S,5.263158,11.111111,17.647059
1321636,21531,21535,5.263158,11.111111,17.647059
285507,20719,20724,5.263158,11.111111,17.647059
2752672,22355,22356,5.263158,11.111111,17.647059
4888155,84997C,84997D,5.263158,11.111111,17.647059
3589145,22865,22866,5.263158,11.111111,17.647059
4885567,84997B,84997C,5.263158,11.111111,17.647059


In [20]:
# ============================================================
# FINAL DISCOUNT STRATEGY
# ============================================================

final_bundles["Discount_Strategy"] = np.where(
    final_bundles["Business_Action"] == "Test Bundle Offer",
    "Test 5% Bundle Discount",
    np.where(
        final_bundles["Business_Action"] == "Cross-Selling Opportunity",
        "Cross-Sell Without Discount",
        "Monitor - No Discount"
    )
)

display(
    final_bundles[
        [
            "Product_A",
            "Product_B",
            "Business_Action",
            "Discount_Strategy",
            "Lift",
            "Combined_Revenue",
            "Growth_Needed_5_Percent"
        ]
    ].head(20)
)

,Product_A,Product_B,Business_Action,Discount_Strategy,Lift,Combined_Revenue,Growth_Needed_5_Percent
910028,21231,21232,Test Bundle Offer,Test 5% Bundle Discount,6.203040,51293.78,5.263158
288778,20723,20724,Cross-Selling Opportunity,Cross-Sell Without Discount,8.923867,21876.91,5.263158
522568,20971,20972,Cross-Selling Opportunity,Cross-Sell Without Discount,10.503150,31090.64,5.263158
4837451,84970L,84970S,Cross-Selling Opportunity,Cross-Sell Without Discount,9.761539,25355.04,5.263158
1321636,21531,21535,Cross-Selling Opportunity,Cross-Sell Without Discount,11.374011,17492.79,5.263158
285507,20719,20724,Cross-Selling Opportunity,Cross-Sell Without Discount,8.115996,25290.31,5.263158
2752672,22355,22356,Cross-Selling Opportunity,Cross-Sell Without Discount,8.679534,27156.19,5.263158
4888155,84997C,84997D,Cross-Selling Opportunity,Cross-Sell Without Discount,11.716858,38968.61,5.263158
3589145,22865,22866,Cross-Selling Opportunity,Cross-Sell Without Discount,11.753115,31565.90,5.263158
4885567,84997B,84997C,Cross-Selling Opportunity,Cross-Sell Without Discount,11.203455,37349.42,5.263158


In [21]:
# ============================================================
# SAVE FINAL BUNDLE RECOMMENDATIONS
# ============================================================

final_bundle_recommendations = final_bundles[
    [
        "Product_A",
        "Product_B",
        "Support",
        "Confidence_A_to_B",
        "Lift",
        "Combined_Revenue",
        "Bundle_Score",
        "Bundle_Recommendation",
        "Business_Action"
    ]
].copy()

final_bundle_recommendations = final_bundle_recommendations.sort_values(
    "Bundle_Score",
    ascending=False
)

display(
    final_bundle_recommendations.head(20)
)

final_bundle_recommendations.to_csv(
    "final_bundle_recommendations.csv",
    index=False
)

print("final_bundle_recommendations.csv saved successfully!")

,Product_A,Product_B,Support,Confidence_A_to_B,Lift,Combined_Revenue,Bundle_Score,Bundle_Recommendation,Business_Action
910028,21231,21232,0.023195,0.485514,6.203040,51293.78,0.069855,Strong Bundle Candidate,Test Bundle Offer
288778,20723,20724,0.014222,0.439528,8.923867,21876.91,0.055784,Strong Bundle Candidate,Cross-Selling Opportunity
522568,20971,20972,0.012791,0.408537,10.503150,31090.64,0.054883,Strong Bundle Candidate,Cross-Selling Opportunity
4837451,84970L,84970S,0.012981,0.423017,9.761539,25355.04,0.053604,Strong Bundle Candidate,Cross-Selling Opportunity
1321636,21531,21535,0.010738,0.417440,11.374011,17492.79,0.050985,Strong Bundle Candidate,Cross-Selling Opportunity
285507,20719,20724,0.014556,0.399738,8.115996,25290.31,0.047225,Strong Bundle Candidate,Cross-Selling Opportunity
2752672,22355,22356,0.015081,0.356659,8.679534,27156.19,0.046686,Strong Bundle Candidate,Cross-Selling Opportunity
4888155,84997C,84997D,0.010643,0.354531,11.716858,38968.61,0.044210,Strong Bundle Candidate,Cross-Selling Opportunity
3589145,22865,22866,0.010404,0.343849,11.753115,31565.90,0.042047,Strong Bundle Candidate,Cross-Selling Opportunity
4885567,84997B,84997C,0.010738,0.336323,11.203455,37349.42,0.040462,Strong Bundle Candidate,Cross-Selling Opportunity


final_bundle_recommendations.csv saved successfully!
